In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse

""" new fn: count the number of nodes in the implementation"""
def count_nodes(psi):
    return np.sum(np.abs(np.diff(np.sign(psi))) / 2)

# creates the matrices necessary for CN
def create_Apl_Ami_Heff_M2_matrices(x_vec, V_vec, Z, ell, deltaTau):
    Delta_x = x_vec[1] - x_vec[0]
    N = len(x_vec)
    
    D_lower = np.ones(N-1, dtype=complex)/(Delta_x**2)
    D_upper = np.ones(N-1, dtype=complex)/(Delta_x**2)
    D_diag = -2*np.ones(N, dtype=complex)/(Delta_x**2)
    
    D = get_sparseMatrix(D_lower, D_diag, D_upper)
    
    M_2_lower = -np.ones(N-1, dtype=complex)/6
    M_2_diag = -10*np.ones(N, dtype=complex)/6
    M_2_upper = -np.ones(N-1, dtype=complex)/6
    
    M_2 = get_sparseMatrix(M_2_lower, M_2_diag, M_2_upper)
    
    V_matrix = sparse.spdiags([V_vec], [0], N, N)
    
    # Hamiltonian (effective)
    H_2 = D + M_2@V_matrix
    
    # CN A matrices
    A_pl = M_2 + 0.5j * deltaTau * H_2
    A_mi = M_2 - 0.5j * deltaTau * H_2
    return A_pl, A_mi, H_2, M_2

def get_sparseMatrix(upper, mid, lower):
    return sparse.diags([lower, mid, upper], offsets=[-1, 0, 1], format="csr")

# Crank-Nicolson propagation
def CN_propagation(A_pl, A_mi, psi, r, N_iter=1000):
    A_pl_lower = get_lower(A_pl)
    A_pl_diag = get_diag(A_pl)
    A_pl_upper = get_upper(A_pl)
    
    for i in (range(N_iter)): # Iterate for imaginary time
        psi_new = thomas_algorithm(A_pl_lower, A_pl_diag, A_pl_upper, A_mi @ psi)
        psi = normalize(psi_new, r)
    return psi

# Compute energy (expectation value of Hamiltonian)
def calc_E_withNumerov(psi, H_2, M_2, r):
    return project(psi, H_2 @ psi, r)/project(psi, M_2 @ psi, r)


def get_diag(mat):
    return np.array([mat[i, i] for i in range(mat.shape[0])], dtype=complex)
def get_upper(mat):
    return np.array([mat[i-1, i] for i in range(1, mat.shape[0])], dtype=complex)
def get_lower(mat):
    return np.array([mat[i, i-1] for i in range(1, mat.shape[0])], dtype=complex)

# general scalar projection function
def project(psi1, psi2, x_vec):
    return np.trapz(np.conj(psi1)*psi2, x=x_vec)
def normalize(psi, x_vec):
    return psi/np.sqrt(project(psi, psi, x_vec))

def thomas_algorithm(a, b, c, d):
    n = len(b)
    
    # Create copies of the input arrays to modify during elimination
    bp = np.copy(b).astype(complex) # diagonal copy
    dp = np.copy(d).astype(complex) # Right-hand side copy
    
    # Forward elimination
    for i in range(1, n):
        # Modify the coefficients in the sub-diagonal and right-hand side
        w = a[i-1] / bp[i-1]
        bp[i] = b[i] - w * c[i-1]
        dp[i] = d[i] - w * dp[i-1]
    
    # Back substitution
    x = np.zeros(n, dtype=complex)
    x[-1] = dp[-1] / bp[-1]
    for i in range(n-2, -1, -1):
        x[i] = (dp[i] - c[i] * x[i+1]) / bp[i]
    return x

In [ ]:
# Constants
Z = 1
ell = 0 # for the lowest hydrogen state 1s
desiredNodes = 1 - ell - 1 # n - l - 1

# Imaginary time step (Δτ)
delta = -0.9
E_k = -0.5
Delta_tau = 2*delta/E_k
imagDeltaTau = (-1j*Delta_tau)

# grid details
Delta_x = 0.2
N = 500
x_vec = np.linspace(-50, 50, N) 

# involves tuning the softcore parameter 'a' to fit the 1s state of hydrogen
maxBisectionIter = 50
a_lower = 0
a_upper = 2


# bisection loop to tune the alpha value until the actual value of energy is achieved
for iter in range(maxBisectionIter):
    a = (a_lower + a_upper) / 2
    print(iter, a)
    
    # Modify the potential for Hydrogen, with the current value of 'a'
    V_softcore_vec = -1/np.sqrt(x_vec**2 + a**2)
    
    # Create matrices
    A_pl, A_mi, H_2, M_2 = create_Apl_Ami_Heff_M2_matrices(x_vec, V_softcore_vec, Z, ell, imagDeltaTau)
    
    # Initial wavefunction (guessed)
    psi = np.random.random(N).astype(complex) # np.zeros(N)
    psi = normalize(psi, x_vec)
    
    # Crank-Nicolson propagation
    psi = CN_propagation(A_pl, A_mi, psi, x_vec, N_iter=1000)
    E = calc_E_withNumerov(psi, H_2, M_2, x_vec)
    noNodes = count_nodes(psi)
    
    # tuning the alpha for next iteration, based on the number of nodes
    if noNodes > desiredNodes:
        a_lower = a
    elif noNodes < desiredNodes:
        a_upper = a
    else:
        if np.abs(E - E_k) < 1e-6:
            break
        elif E < E_k:
            a_lower = a
        else:
            a_upper = a
            


0 1.0
1 1.5
2 1.25
3 1.375
4 1.4375
5 1.40625
6 1.421875
7 1.4140625
8 1.41796875
9 1.416015625
10 1.4150390625
11 1.41455078125
12 1.414306640625
13 1.4141845703125


In [ ]:
plt.plot(x_vec, psi, label=f"Energy={np.round(np.real(E), 8)}")
plt.xlabel("x(atomic units)")
plt.grid()
plt.ylabel("Wavefunction")
plt.legend()
plt.title(f"H-atom wavefunction : $\psi_0(x)$ \n Softcore Parameter={a}")

np.save("a.npy", a)
np.save("psi0.npy", psi)

<>:6: SyntaxWarning: invalid escape sequence '\p'
<>:6: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_9788/3244651209.py:6: SyntaxWarning: invalid escape sequence '\p'
  plt.title(f"H-atom wavefunction : $\psi_0(x)$ \n Softcore Parameter={a}")
/tmp/ipykernel_9788/3244651209.py:6: SyntaxWarning: invalid escape sequence '\p'
  plt.title(f"H-atom wavefunction : $\psi_0(x)$ \n Softcore Parameter={a}")


NameError: name 'plt' is not defined